## Future idea: PySpark processing on the full dataset
PySpark uses lazy evaluation by default for all data transformations.
Purpose:
- filter observations for European weather stations,
- select relevant weather metrics,
- perform yearly/monthly/station-level aggregations,
- compare PySpark transformations with Pandas and Polars workflows,
- practice Spark DataFrame syntax for scalable data processing.

- implement a similar weather processing workflow in PySpark
- practice Spark DataFrame transformations
- work with lazy execution and query plans
- perform filtering, joins and aggregations on the full dataset

This notebook should demonstrate how the same data processing logic can be expressed in PySpark and how Spark can be used for larger datasets or distributed processing scenarios.

### Spark setup

In [4]:
# Import libraries
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pathlib import Path

# Initalize the Spark Session which is the main entry point for working with Spark
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('Weather Processing')
    .getOrCreate()
)

# Paths to the full Parquet and CSV datasets stored outside the repository
WEATHER = Path(r"C:\Users\zychl\Desktop\Data Engineering\weather_data_processing\00_raw_data\weather.parquet")
STATIONS = Path(r"C:\Users\zychl\Desktop\Data Engineering\weather_data_processing\00_raw_data\stations.csv")

#### Define `stations` schema

The schema for `stations.csv` was defined manually based on metadata from the existing PostgreSQL `bronze.stations` table.

Column types and nullability were inspected using:

```sql
SELECT
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'bronze'
  AND table_name = 'stations';

In [5]:
# Define the schema based on the existing PostgreSQL table metadata and observed nullability
schema = T.StructType([
    T.StructField('station', T.StringType(), False),
    T.StructField('latitude', T.DoubleType(), False),
    T.StructField('longitude', T.DoubleType(), False),
    T.StructField('elevation', T.DoubleType(), False),
    T.StructField('state', T.StringType(), True),
    T.StructField('station_name', T.StringType(), False),
    T.StructField('gsn_flag', T.StringType(), True),
    T.StructField('hcn_flag', T.StringType(), True),
    T.StructField('wmo_id', T.DoubleType(), True)
])

#### Load datasets

In [ ]:
# Read weather.parquet and stations.csv files.
# Convert the Path object to a string because Spark expects the file path as text
df_raw_weather = (
    spark.read
    .parquet(str(WEATHER))
    )

df_raw_stations = (
    spark.read
    .option('header', True)
    .schema(schema)
    .csv(str(STATIONS))
    )

# Show DataFrames
{
    'weather': df_raw_weather.show(5), 
    'stations': df_raw_stations.show(5)
}

#### Inspect DataFrames

The schemas below are inspected to verify column names, data types, and nullability.

**Spark and Polars data type mapping**

A quick reference for equivalent data types used in Spark and Polars.

| Spark type | Polars type |
|---|---|
| `string` | `String` |
| `long` | `Int64` |
| `double` | `Float64` |
| `boolean` | `Boolean` |
| `date` | `Date` |
| `timestamp` | `Datetime` |

In [27]:
# Inspect DataFrame schemas
df_raw_weather.printSchema()
df_raw_stations.printSchema()

root
 |-- station: string (nullable = true)
 |-- observation_date: long (nullable = true)
 |-- metric: string (nullable = true)
 |-- value: long (nullable = true)
 |-- measurement_flag: string (nullable = true)
 |-- quality_flag: string (nullable = true)
 |-- source_flag: string (nullable = true)
 |-- observation_time: double (nullable = true)

root
 |-- station: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- elevation: string (nullable = true)
 |-- state: string (nullable = true)
 |-- station_name: string (nullable = true)
 |-- gsn_flag: string (nullable = true)
 |-- hcn_flag: string (nullable = true)
 |-- wmo_id: string (nullable = true)

